In [1]:
import pandas as pd
import os
import datetime
import pytz

# Step 1. Load in the two datasets (Newest and Oldest)

In [2]:
print(os.getcwd())

/Users/rishabhbaral/Documents/ASU/Classes/Spring25/CSE576-NLP/Course_Project/SportsT2T/LiveSum_++


In [3]:
newest = pd.read_csv('Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2024-25.csv')
oldest = pd.read_csv('Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2013-14.csv')

# Step 1b. Print the two dataframes (both first few rows and shape to understand how to combine them)

In [4]:
print(newest.head(5))
print(oldest.head(5))
print(newest.shape)
print(oldest.shape)

      Season                                          Match URL  \
0  2024-2025  https://www.flashscore.com/match/football/44RK...   
1  2024-2025  https://www.flashscore.com/match/football/QJFT...   
2  2024-2025  https://www.flashscore.com/match/football/hQ7F...   
3  2024-2025  https://www.flashscore.com/match/football/lh1O...   
4  2024-2025  https://www.flashscore.com/match/football/pOnf...   

        Home Team        Away Team Match Date Time  1H_Home_Shots_on_target  \
0     Bournemouth        Leicester    25.05. 08:00                      3.0   
1          Fulham  Manchester City    25.05. 08:00                      2.0   
2         Ipswich         West Ham    25.05. 08:00                      2.0   
3       Liverpool   Crystal Palace    25.05. 08:00                      1.0   
4  Manchester Utd      Aston Villa    25.05. 08:00                      5.0   

   1H_Away_Shots_on_target  1H_Home_Shots_off_target  \
0                      0.0                       3.0   
1         

# Step 2. Combine the two into one larger dataframe

In [5]:
old_v_new = pd.concat([oldest, newest], ignore_index=True)

In [6]:
print(old_v_new.shape) # this should be same number of columns, # but double the number of rows
print(old_v_new.head(5))

(760, 79)
      Season                                          Match URL  \
0  2013-2014  https://www.flashscore.com/match/football/nch9...   
1  2013-2014  https://www.flashscore.com/match/football/tnbW...   
2  2013-2014  https://www.flashscore.com/match/football/fk2z...   
3  2013-2014  https://www.flashscore.com/match/football/bNSG...   
4  2013-2014  https://www.flashscore.com/match/football/0WEr...   

         Home Team       Away Team Match Date Time  1H_Home_Shots_on_target  \
0          Cardiff         Chelsea    11.05. 07:00                      2.0   
1           Fulham  Crystal Palace    11.05. 07:00                      2.0   
2             Hull         Everton    11.05. 07:00                      1.0   
3        Liverpool    Newcastle\n2    11.05. 07:00                      2.0   
4  Manchester City        West Ham    11.05. 07:00                      3.0   

   1H_Away_Shots_on_target  1H_Home_Shots_off_target  \
0                      2.0                       2.0   


# Step 3. Clean up the data to add new information and delete unneccessary information

In [7]:
old_v_new.drop(columns = ['Match URL'], inplace=True) # drop the Match URL column

In [8]:
import pandas as pd
from datetime import datetime

# Step 1: Check dtype
print(old_v_new['Match Date Time'].dtype)

# Step 2: Convert to string
old_v_new['Match Date Time'] = old_v_new['Match Date Time'].astype(str)
print(old_v_new['Match Date Time'].dtype)

# Step 3: Display first 5 entries for reference
print(old_v_new['Match Date Time'].head(5))

# Step 4: Reset index to use row numbers for year logic
old_v_new = old_v_new.reset_index(drop=True)

# Step 5: Convert to datetime without year
old_v_new['partial_dt'] = pd.to_datetime(old_v_new['Match Date Time'], format="%d.%m. %H:%M")

# Step 6: Apply year logic
def determine_year(row):
    idx = row.name  # .name gives index value when axis=1
    month = row['partial_dt'].month
    if idx <= 379:
        return 2013 if month >= 8 else 2014
    else:
        return 2024 if month >= 8 else 2025

old_v_new['year'] = old_v_new.apply(determine_year, axis=1)

# Step 7: Combine to full datetime
old_v_new['final_datetime'] = old_v_new.apply(
    lambda row: datetime(
        year=row['year'],
        month=row['partial_dt'].month,
        day=row['partial_dt'].day,
        hour=row['partial_dt'].hour,
        minute=row['partial_dt'].minute
    ),
    axis=1
)

# Step 8: Drop intermediate columns
old_v_new = old_v_new.drop(columns=['partial_dt', 'year'])

# Optional: If you want to overwrite the original column
old_v_new['Match Date Time'] = old_v_new['final_datetime']
old_v_new = old_v_new.drop(columns=['final_datetime'])

object
object
0    11.05. 07:00
1    11.05. 07:00
2    11.05. 07:00
3    11.05. 07:00
4    11.05. 07:00
Name: Match Date Time, dtype: object


In [9]:
# Step 9: Reformat the Match Date Time column to the desired format, after applying the timezone (should all be Europe/London)
old_v_new['Match Date Time'] = old_v_new['Match Date Time'].dt.tz_localize('America/Phoenix').dt.tz_convert('Europe/London').dt.strftime('%Y/%m/%d %H:%M')

In [10]:
#Step 10: Final check
display(old_v_new.head())

,Season,Home Team,Away Team,Match Date Time,1H_Home_Shots_on_target,1H_Away_Shots_on_target,1H_Home_Shots_off_target,1H_Away_Shots_off_target,1H_Home_Blocked_Shots,1H_Away_Blocked_Shots,...,FT_Home_Corner_Kicks,FT_Away_Corner_Kicks,FT_Home_Throwins,FT_Away_Throwins,FT_Home_Free_Kicks,FT_Away_Free_Kicks,FT_Home_Goalkeeper_Saves,FT_Away_Goalkeeper_Saves,First Half Commentary,Second Half Commentary
0,2013-2014,Cardiff,Chelsea,2014/05/11 15:00,2.0,2.0,2.0,6.0,2.0,3.0,...,3,8,19,24,11,8,5,3,45+3'\nWe have seen an attractive offensive ga...,"90+4'\nAccording to the statistics, the match ..."
1,2013-2014,Fulham,Crystal Palace,2014/05/11 15:00,2.0,4.0,4.0,2.0,2.0,3.0,...,6,4,22,17,16,12,4,3,45+2'\nThe game produced by the players until ...,"90+5'\nWe have seen a great game today, let's ..."
2,2013-2014,Hull,Everton,2014/05/11 15:00,1.0,2.0,3.0,4.0,1.0,0.0,...,6,4,11,17,10,10,2,3,45+2'\nWe have been witnesses to an ordinary g...,90+3'\nThe performance from both sides could b...
3,2013-2014,Liverpool,Newcastle\n2,2014/05/11 15:00,2.0,2.0,1.0,3.0,2.0,2.0,...,6,2,13,18,18,11,1,3,45+3'\nIt wasn't the most inspiring 45 minutes...,"90+5'\nWe saw a few chances and nice plays, bu..."
4,2013-2014,Manchester City,West Ham,2014/05/11 15:00,3.0,0.0,7.0,1.0,4.0,0.0,...,12,1,17,17,10,9,0,5,45+2'\nNot the best game in the world so far b...,90+4'\nAll in all that was an entertaining 90 ...


# Step 3. Save to a CSV file with a descriptive name

In [11]:
display(old_v_new)

,Season,Home Team,Away Team,Match Date Time,1H_Home_Shots_on_target,1H_Away_Shots_on_target,1H_Home_Shots_off_target,1H_Away_Shots_off_target,1H_Home_Blocked_Shots,1H_Away_Blocked_Shots,...,FT_Home_Corner_Kicks,FT_Away_Corner_Kicks,FT_Home_Throwins,FT_Away_Throwins,FT_Home_Free_Kicks,FT_Away_Free_Kicks,FT_Home_Goalkeeper_Saves,FT_Away_Goalkeeper_Saves,First Half Commentary,Second Half Commentary
0,2013-2014,Cardiff,Chelsea,2014/05/11 15:00,2.0,2.0,2.0,6.0,2.0,3.0,...,3,8,19,24,11,8,5,3,45+3'\nWe have seen an attractive offensive ga...,"90+4'\nAccording to the statistics, the match ..."
1,2013-2014,Fulham,Crystal Palace,2014/05/11 15:00,2.0,4.0,4.0,2.0,2.0,3.0,...,6,4,22,17,16,12,4,3,45+2'\nThe game produced by the players until ...,"90+5'\nWe have seen a great game today, let's ..."
2,2013-2014,Hull,Everton,2014/05/11 15:00,1.0,2.0,3.0,4.0,1.0,0.0,...,6,4,11,17,10,10,2,3,45+2'\nWe have been witnesses to an ordinary g...,90+3'\nThe performance from both sides could b...
3,2013-2014,Liverpool,Newcastle\n2,2014/05/11 15:00,2.0,2.0,1.0,3.0,2.0,2.0,...,6,2,13,18,18,11,1,3,45+3'\nIt wasn't the most inspiring 45 minutes...,"90+5'\nWe saw a few chances and nice plays, bu..."
4,2013-2014,Manchester City,West Ham,2014/05/11 15:00,3.0,0.0,7.0,1.0,4.0,0.0,...,12,1,17,17,10,9,0,5,45+2'\nNot the best game in the world so far b...,90+4'\nAll in all that was an entertaining 90 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,2024-2025,Everton,Brighton,2024/08/17 15:00,1.0,1.0,5.0,4.0,0.0,0.0,...,1,5,31,16,9,9,2,1,45+5'\nThe end of the first half.\n45+4'\nThe ...,90+11'\nThere will be no more action in this m...
756,2024-2025,Newcastle,Southampton,2024/08/17 15:00,1.0,0.0,0.0,3.0,2.0,1.0,...,3,12,12,19,15,15,5,0,45+5'\nThe first half of today's match has jus...,90+7'\nThe referee blows his whistle for the f...
757,2024-2025,Nottingham,Bournemouth,2024/08/17 15:00,5.0,2.0,2.0,5.0,2.0,0.0,...,2,6,27,19,9,19,4,7,45+13'\nThe first half of today's match has ju...,90+6'\nThe match has just finished.\n90+5'\nTh...
758,2024-2025,Ipswich,Liverpool,2024/08/17 12:30,2.0,0.0,1.0,1.0,1.0,2.0,...,2,10,14,24,19,10,3,2,45+2'\nThe referee blows his whistle to end th...,90+9'\nThere will be no more action in this ma...


In [12]:
#1. Replace all NaN values in Commentary columns with 'Commentary Not Available'
old_v_new.fillna('Commentary Not Available', inplace=True)
#2. Clean the team names in the Home and away columns (i.e. remove any numbers (Newcastle\n2))
old_v_new['Home Team'] = old_v_new['Home Team'].str.replace(r'\d+', '', regex=True).str.strip()
old_v_new['Away Team'] = old_v_new['Away Team'].str.replace(r'\d+', '', regex=True).str.strip()

/var/folders/yv/cngmzv7n1y14gnt9f3njv76c0000gn/T/ipykernel_12309/4294812157.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Commentary Not Available' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  old_v_new.fillna('Commentary Not Available', inplace=True)


In [13]:
display(old_v_new.head(10))

,Season,Home Team,Away Team,Match Date Time,1H_Home_Shots_on_target,1H_Away_Shots_on_target,1H_Home_Shots_off_target,1H_Away_Shots_off_target,1H_Home_Blocked_Shots,1H_Away_Blocked_Shots,...,FT_Home_Corner_Kicks,FT_Away_Corner_Kicks,FT_Home_Throwins,FT_Away_Throwins,FT_Home_Free_Kicks,FT_Away_Free_Kicks,FT_Home_Goalkeeper_Saves,FT_Away_Goalkeeper_Saves,First Half Commentary,Second Half Commentary
0,2013-2014,Cardiff,Chelsea,2014/05/11 15:00,2.0,2.0,2.0,6.0,2.0,3.0,...,3,8,19,24,11,8,5,3,45+3'\nWe have seen an attractive offensive ga...,"90+4'\nAccording to the statistics, the match ..."
1,2013-2014,Fulham,Crystal Palace,2014/05/11 15:00,2.0,4.0,4.0,2.0,2.0,3.0,...,6,4,22,17,16,12,4,3,45+2'\nThe game produced by the players until ...,"90+5'\nWe have seen a great game today, let's ..."
2,2013-2014,Hull,Everton,2014/05/11 15:00,1.0,2.0,3.0,4.0,1.0,0.0,...,6,4,11,17,10,10,2,3,45+2'\nWe have been witnesses to an ordinary g...,90+3'\nThe performance from both sides could b...
3,2013-2014,Liverpool,Newcastle,2014/05/11 15:00,2.0,2.0,1.0,3.0,2.0,2.0,...,6,2,13,18,18,11,1,3,45+3'\nIt wasn't the most inspiring 45 minutes...,"90+5'\nWe saw a few chances and nice plays, bu..."
4,2013-2014,Manchester City,West Ham,2014/05/11 15:00,3.0,0.0,7.0,1.0,4.0,0.0,...,12,1,17,17,10,9,0,5,45+2'\nNot the best game in the world so far b...,90+4'\nAll in all that was an entertaining 90 ...
5,2013-2014,Norwich,Arsenal,2014/05/11 15:00,1.0,3.0,2.0,3.0,2.0,1.0,...,4,4,19,23,7,9,6,5,45+2'\nWe are seeing rather average performanc...,90+3'\nThe fans were presented with a dull gam...
6,2013-2014,Southampton,Manchester Utd,2014/05/11 15:00,5.0,0.0,4.0,2.0,3.0,1.0,...,6,2,21,21,11,17,1,5,45+3'\nIt wasn't the most inspiring 45 minutes...,90+4'\nThere wasn't a lot of action on the pit...
7,2013-2014,Sunderland,Swansea,2014/05/11 15:00,1.0,2.0,4.0,1.0,4.0,2.0,...,6,3,28,21,16,14,1,3,45+2'\nThis is how football should be - full o...,90+4'\nIt was a very watchable match. Great go...
8,2013-2014,Tottenham,Aston Villa,2014/05/11 15:00,6.0,0.0,3.0,1.0,2.0,1.0,...,5,1,18,25,10,17,1,3,45+3'\nThere was plenty of entertainment durin...,90+3'\nWe have been watching a lacklustre game...
9,2013-2014,West Brom,Stoke,2014/05/11 15:00,2.0,1.0,4.0,3.0,6.0,3.0,...,11,6,16,18,10,5,2,3,45+2'\nThere was plenty of entertainment durin...,90+4'\nWe were witnesses to an uneventful game...


In [14]:
old_v_new.to_csv('flashscore_old_vs_new.csv', index=False)